# Section 7 · Observability 1 — Console Tracing (no external services) (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

Strands emits OpenTelemetry (OTel) spans for every agent invocation, reasoning cycle, model call and tool call. The simplest way to see them: print them to the console.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv(override=True)  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
API_KEY = os.getenv("GROQ_API_KEY")
BASE_URL = os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1")  # or a local OpenAI-compatible server

model = OpenAIModel(
    client_args={
        "api_key": API_KEY,
        "base_url": BASE_URL,  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    # params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")
print(f"Using {BASE_URL} as the base URL")


Using openai/gpt-oss-120b on Groq
Using https://api.groq.com/openai/v1 as the base URL


In [6]:
from strands import Agent
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator

# Print every span to stdout
strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

agent = Agent(model=model, system_prompt="You are a helpful assistant.", tools=[calculator], callback_handler=None)

response = agent("What is 15% of 2,847? Then explain in two sentences why tracing matters for AI agents.")
print("\n\nANSWER:\n", response)

Overriding of current TracerProvider is not allowed


{
    "name": "chat",
    "context": {
        "trace_id": "0xfd614ab7d611f997f89c2f7c519d15d7",
        "span_id": "0x610c2f25619141bd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x42920b1c56770673",
    "start_time": "2026-11-01T08:01:02.821931Z",
    "end_time": "2026-11-01T08:01:04.144580Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-11-01T08:01:02.821933+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "gen_ai.request.model": "openai/gpt-oss-120b",
        "gen_ai.event.end_time": "2026-11-01T08:01:04.144523+00:00",
        "gen_ai.usage.prompt_tokens": 1307,
        "gen_ai.usage.input_tokens": 1307,
        "gen_ai.usage.completion_tokens": 173,
        "gen_ai.usage.output_tokens": 173,
        "gen_ai.usage.total_tokens": 1480,
        "gen_ai.server.time_to_first_token": 975
    },
    "events": [
        {
       

DEPRECATION WARNING: calculator is deprecated. This warning becomes an import error in v0.9.0. To achieve similar functionality, use the shell tool vended by strands-agents (from strands.vended_tools import shell). This does change the security boundary: calculator only ever evaluated an expression checked against an AST allowlist, while shell executes arbitrary commands, so review it against your threat model before switching.


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 2847*0.15           │                                                                            │
│  │ Result    │ 427.05              │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
    "name": "execute_tool calculator",
    "context": {
        "trace_id": "0xfd614ab7d611f997f89c2f7c519d15d7",
        "span_id": "0x3dbf0c93374ec954",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x42920b1c56770673",
    "start_time": "2026-11-01T08:01:04.151605Z",
    "end_time": "2026-11-01T08:01:04.186085Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-11-01T08:01:04.151611+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "calculator",
        "gen_ai.tool.call.id": "fc_fa81b834-8405-47bb-ae21-6315480c24b1",
        "gen_ai.tool.description": "Calculator powered by SymPy for comprehensive mathematical operations.\n\nThis tool provides advanced mathematical functionality through multiple operation modes,\nincluding expression evaluation, equation solving, calculus operations (derivatives, integ

count=<1> | reasoningContent is not supported in multi-turn conversations with the Chat Completions API


{
    "name": "chat",
    "context": {
        "trace_id": "0xfd614ab7d611f997f89c2f7c519d15d7",
        "span_id": "0x2612adc46549be71",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x97db9661e7aa9a85",
    "start_time": "2026-11-01T08:01:04.189475Z",
    "end_time": "2026-11-01T08:01:04.931787Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-11-01T08:01:04.189484+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "gen_ai.request.model": "openai/gpt-oss-120b",
        "gen_ai.event.end_time": "2026-11-01T08:01:04.931745+00:00",
        "gen_ai.usage.prompt_tokens": 1349,
        "gen_ai.usage.input_tokens": 1349,
        "gen_ai.usage.completion_tokens": 109,
        "gen_ai.usage.output_tokens": 109,
        "gen_ai.usage.total_tokens": 1458,
        "gen_ai.server.time_to_first_token": 509
    },
    "events": [
        {
       

In the output above, look for:
- an **`invoke_agent`** span (top level) with total token counts
- **`execute_event_loop_cycle`** spans — one per reasoning loop
- **`chat`** spans — each model call, with input/output tokens and latency
- **`execute_tool calculator`** — the tool call with its parameters and result